# Fine-tuning de FLAN-T5 como tutor de matemáticas integrado

### Notebook 4 de 4 · Arquitectura encoder-decoder (comprensión + generación)

**Proyecto:** Tutor inteligente de matemáticas · Comparación de arquitecturas
mediante fine-tuning
**Curso:** SI4006 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Notebook base:** `S04_Lab_Fine_tuning_Qwen.ipynb`

---

## 1 · Introducción

### La propuesta

Los notebooks 2 y 3 dividieron el problema: un encoder entiende y clasifica, un
decoder genera. FLAN-T5 propone hacerlo todo en **una sola arquitectura** que
tiene ambas partes por diseño.

Y lo hacemos explícito en la salida: entrenamos al modelo para que escriba la
categoría **antes** del procedimiento.

```
Entrada:  "resuelve el problema: En una escuela hay 420 estudiantes.
           Si el 25% participa en un torneo, cuantos participan?"

Salida:   "Categoria: porcentajes
           Paso 1: El 25% equivale a multiplicar por 0.25.
           Paso 2: 420 x 0.25 = 105.
           Respuesta final: 105 estudiantes"
            ^^^^^^^^^^^^^^^^^^^^^^^^^^
            clasificación + generación en una sola pasada
```

Esto convierte al notebook en el **contraste directo del pipeline del notebook
3**: los mismos dos trabajos, resueltos por un modelo en lugar de por dos, y
medidos con las mismas métricas sobre los mismos datos. La comparación de
ambos es el núcleo del experimento.

### Por qué encoder-decoder

| Componente | Papel |
|---|---|
| **Encoder** | Lee el enunciado completo con atención bidireccional, igual que BERT. Construye una representación de todo el problema antes de escribir nada. |
| **Cross-attention** | En cada paso de generación, el decoder consulta la representación completa del enunciado. No depende de "recordarlo" a través de su propio contexto. |
| **Decoder** | Genera el procedimiento token a token, igual que Qwen. |

Un decoder-only como Qwen procesa el enunciado con la misma máquina causal con
la que escribe la respuesta: al leer "420" todavía no sabe que después vendrá
"25%". El encoder de T5 no tiene esa limitación. Para problemas donde el dato
clave aparece al final, esa es una ventaja arquitectónica real.

### Por qué `google/flan-t5-base`

- 250M de parámetros: entre BETO (110M) y Qwen (1.5B). Entrena rápido.
- El sufijo **FLAN** significa que fue afinado con instrucciones sobre más de
  1.800 tareas. Sigue instrucciones sin necesidad de tokens de chat especiales.
- Es el encoder-decoder abierto de referencia: la comparación es reproducible y
  el profesor puede contrastarla con literatura publicada.

### Ventajas y limitaciones

**Ventajas**
- Un solo modelo que versionar, desplegar y monitorizar. No hay riesgo de
  desalineación entre componentes.
- Sin propagación de error en cascada: clasificación y generación se optimizan
  con la misma pérdida y son coherentes por construcción.
- Comprensión bidireccional del enunciado, que un decoder-only no tiene.

**Limitaciones — y una es seria**
- **El tokenizador de T5 fue construido para inglés.** Su SentencePiece de 32k
  no cubre bien las tildes, la apertura de interrogación `¿`, ni los símbolos
  `×`, `÷`, `√`, `²`. Sobre un corpus 100% en español y lleno de notación
  matemática, esto no es un detalle: es el factor limitante del notebook. La
  sección 6 lo mide y lo trata.
- Salida de longitud acotada; menos flexible que un decoder grande.
- Su preentrenamiento matemático es más débil que el de Qwen.

## 2 · Objetivos

1. **Diagnosticar el tokenizador** sobre nuestro corpus: medir la tasa de
   tokens `<unk>` y determinar qué caracteres se pierden.
2. Diseñar y aplicar una **normalización** del texto que preserve el
   significado matemático dentro del vocabulario disponible.
3. Entrenar FLAN-T5 con LoRA para producir categoría + procedimiento +
   respuesta en una sola secuencia.
4. Evaluar **las dos capacidades por separado**: exactitud de la respuesta
   (comparable con los notebooks 1 y 3) y accuracy de clasificación
   (comparable con el notebook 2).
5. Contrastar el modelo integrado contra el pipeline modular y dejar los
   resultados listos para la comparación final.

## 0 · Preparación del entorno

Instalamos el ecosistema Hugging Face. Las versiones se fijan por rango mayor
para evitar que un cambio de API rompa el notebook meses después.

> **Antes de empezar:** activen la GPU en `Entorno de ejecución → Cambiar tipo
> de entorno de ejecución → T4 GPU`. Sin GPU el entrenamiento es inviable.

**Sobre la desinstalación de `torchao`.** Colab trae `torchao` preinstalado, y
`peft` comprueba su versión con una función que **lanza `ImportError` en lugar
de devolver `False`** cuando la encuentra más antigua de lo que espera. El
resultado es que `get_peft_model()` falla con un error que no tiene ninguna
relación aparente con LoRA.

Ningún notebook del proyecto usa cuantización de torchao, así que lo quitamos.
La alternativa —actualizarlo— también funcionaría, pero torchao está acoplado a
la versión de torch y actualizarlo puede arrastrar un torch distinto y romper
otras cosas en Colab. Desinstalarlo no afecta a nada de lo que hacemos aquí.

In [ ]:
%pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" \
    "accelerate>=0.33" "bitsandbytes>=0.43" "scikit-learn>=1.3" wandb

# Ver la nota de arriba: evita que get_peft_model() falle con un ImportError
# de torchao que nada tiene que ver con LoRA.
%pip uninstall -y -q torchao

print("Librerías instaladas. Si Colab pide reiniciar la sesión, reinícienla y sigan desde aquí.")

In [ ]:
import os, random, sys
import numpy as np
import torch
import transformers

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
transformers.set_seed(SEMILLA)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers  {transformers.__version__}")
print(f"torch         {torch.__version__}")
print(f"Dispositivo   {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU           {torch.cuda.get_device_name(0)}")
else:
    print("AVISO: sin GPU el fine-tuning tardará horas. Activen el runtime T4.")

# Detección temprana del conflicto peft/torchao. Más vale que salte aquí, en la
# celda de entorno, que dentro de get_peft_model() veinte celdas más adelante
# con un mensaje que no menciona LoRA por ninguna parte.
try:
    from peft.import_utils import is_torchao_available
except Exception:
    pass                       # ruta interna de peft cambiada: no es un problema
else:
    try:
        is_torchao_available()
    except ImportError as e:
        print(f"\nAVISO peft/torchao: {e}")
        print("  Solución: ejecuten  %pip uninstall -y torchao  y reinicien la sesión.")

### Persistencia entre notebooks

Los notebooks 3 y 5 **leen carpetas que producen los notebooks 1, 2 y 4**:

```
1 · Qwen    -> adaptadores/qwen-lora/      ─┐
2 · BERT    -> modelos/bert-clasificador/  ─┤-> el notebook 3 las lee
4 · FLAN-T5 -> adaptadores/flan-t5-lora/    │
1,2,3,4     -> resultados/*.json           ─┴-> el notebook 5 los lee
```

En Colab, `/content` se borra al desconectar el runtime, así que ese trabajo se
perdería entre sesiones. Montando Google Drive y trabajando desde una carpeta
suya, los artefactos sobreviven y cada notebook se puede ejecutar el día que se
pueda.

Con `USAR_DRIVE = False` todo queda en `/content`, lo cual es válido si
ejecutan los notebooks 1, 2 y 3 seguidos sin desconectar.

Fuera de Colab la celda no hace nada: el directorio de trabajo se queda como
está.

> **Los checkpoints intermedios nunca van a Drive.** El `Trainer` guarda en
> `output_dir` el modelo *más el estado del optimizador* en cada época. Para el
> notebook 2, que hace fine-tuning completo de BETO, eso son varios GB que
> además se escribirían por red. Como son desechables —lo que importa es el
> modelo final—, se mandan siempre al disco local del runtime mediante
> `DIR_CHECKPOINTS`.

In [ ]:
import os
from pathlib import Path

USAR_DRIVE    = True
CARPETA_DRIVE = "/content/drive/MyDrive/ProyectoIA"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_DRIVE)

# Checkpoints del Trainer: grandes y desechables -> siempre en disco local.
DIR_CHECKPOINTS = "/content/salidas" if EN_COLAB else "salidas"

print(f"En Colab           : {EN_COLAB}")
print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Checkpoints en     : {DIR_CHECKPOINTS}")

## 3 · Arquitectura del modelo

```
   "resuelve el problema: el 25% de 420 estudiantes"
                        |
        ┌───────────────v────────────────┐
        │          ENCODER               │
        │   12 bloques                   │
        │   autoatención BIDIRECCIONAL   │   <- como BERT
        │   -> H (representación         │
        │        del enunciado)          │
        └───────────────┬────────────────┘
                        │  H
                        │        ┌─────────────────────────┐
                        └───────>│       DECODER           │
                                 │   12 bloques            │
                                 │   1. autoatención causal│  <- como Qwen
                                 │   2. CROSS-ATTENTION -> H│  <- lo propio de T5
                                 │   3. feed-forward       │
                                 └────────────┬────────────┘
                                              │
                                "Categoria: porcentajes
                                 Paso 1: ...
                                 Respuesta final: 105"
```

La **cross-attention** es lo que distingue a esta arquitectura. En cada token
que genera, el decoder puede volver a mirar cualquier parte del enunciado
original, ya codificado bidireccionalmente. Qwen tiene que arrastrar esa
información por su propio contexto causal; T5 la tiene disponible siempre.

LoRA se aplica a las proyecciones `q` y `v`, que en T5 aparecen en tres sitios:
la autoatención del encoder, la autoatención del decoder y la cross-attention.
Con `target_modules=["q", "v"]` se cubren los tres.

In [ ]:
MODELO_ID = "google/flan-t5-base"
# Alternativas: "google/flan-t5-small" (80M, más rápido para depurar)
#               "google/flan-t5-large" (780M, no cabe cómodo con LoRA en T4)

DIR_ADAPTADOR = "adaptadores/flan-t5-lora"

LONGITUD_ENTRADA = 128     # se justifica en la sección 6
LONGITUD_SALIDA  = 160
MAX_TOKENS_GEN   = 160

# Si es True, además de normalizar símbolos se eliminan las tildes.
# La sección 6 mide si hace falta.
ELIMINAR_TILDES = True

## 4 · Carga del dataset

### 4.1 · Materialización del corpus

El corpus vive en `data/math_tutor_dataset.jsonl`, generado por
`scripts/dataset_fuente.py`. Para que el notebook funcione en Colab sin subir
archivos, abajo va una **copia comprimida** de ese mismo archivo.

La celda **no sobrescribe** el JSONL si ya existe: eso permite escalar el
dataset (reemplazar el archivo por uno mayor) sin tocar el notebook. El hash
SHA-256 que se imprime debe ser idéntico en los cinco notebooks; si difiere,
alguno está entrenando con datos distintos y la comparación no sería válida.

Hash esperado de la versión embebida: `cf4732834d64a196…`

In [ ]:
import base64, gzip, hashlib, json
from pathlib import Path

RUTA_DATOS = Path("data/math_tutor_dataset.jsonl")
SHA_ESPERADO = "cf4732834d64a196d49eda88ddac8694a529a8bfc6b843e8eb8d74b8c6d59ecc"

_BLOB = (
    "H4sIAAAAAAAC/9V9XY/jSJLYuwH/B2KABrrR3ZL4KaqAQWM8O4DvMLc3t+Pzi20sWBK7mmNJrKWkQlUfDPje7h8s4Ld5nId5"
    "2BsbBublgKkf4P+wv8QRkUkyM5lBJlWqKhWw21OimBmhjMiMz4z4py+K1RcX3he7w+b9zP/iHfx1vS72+GhfZcUWnyyzfX5V"
    "VkUmX8y0h3+kCWbwaF9cl/hKeZ1X2bIoafC2uMnX+PQy2xXLEh/BqKt8K5/l+CTfArAVzf+HfHfI1ze5t868XXF1KOC73JNT"
    "3v+yvfCCKPbeeuE8nRC62boQI7/LdqXnX3jfA4abcucdtvDFKt+981b5Mt9mO+/OW+Js+CeM2ubZdlUCnJ23LD5W2W7yX7c0"
    "R6DA8L70kiCEbwCv60O+22fex2KbrS/wMcK/hhE7AP9fvjgeLs5jQvziv8HDqgb6RwKKv1KCvcnWZaV8znd/XOUbXP6P2XqX"
    "/49//+/+SaVs8CIoG07muAjBJOJo+xUtH64yTLIsNxmucbHJ1l4Gk+3gk4c/o1JIqUyKSzthqDkxyekMCodZgLAEnJgUnDiR"
    "MDwNCa+r8nKdi9fGUvBvD9nWq/JlWVVAMFjZ2Pvvxfr+l02+r2CdrssK12qT3f8l22bA9cEktb2wz6pVPvF++7evD/c/bvfw"
    "hfpSO3++9fblPlt/sPPCN2vxtZcTifC3I1FwX61gnlUBy75dFtq+RpTfEl5IJYYTQhUfnSncYQqu0KHxLNEBqXCHC3NET72/"
    "t4f8phy9wf0gnuEpF6WzgcN7XYoV3d7/uslhTRQaKpPAukbzcGYjIz63n8+dqfE1y6QcteqZGxLVDwZoFL8MGs3EUTabJAyJ"
    "vtktq+KywKWEl/BoLPGPmXedVZkHc2brrBKrLA7MXKVeOz0utD8JYxv18Lmx846AijNY4HGErYE2hK0fDBA2eSmbDw8j0jJS"
    "x913XRW0RdrBX4KKolCzHpDDuZhXS3gXztg5gYBX51bazuP+fckAxUGD4DjSznXCzl3IOn9igWuj6j+CHL0sLtdFuc+XIDnz"
    "7f3PmeeDrrguLlFa3qG8hH0B4jJd1A9ppp0mYeU3e2CL3Ms+lVX2gdOw5IyVPhutOYm+7LABDLN3Xra7/9n700HoQRv9jCZt"
    "FjD60gvCyMYF8FgCMNWu4+HTQa5D5hhCB98wBjx24IxU44wbWsO93LhPyh5fZ5sC9jvSswIO8OPZDA7EfCeWa33Y5sghAcg2"
    "5THorKC/KtzRjB9WuOjN0q4Dye27Ag7VWIGQeiuxwK0M/7VxBH0h0OwoXc5wiQWsEDlOMMA2rEDPHZhhcS7M8BVYnh7+tlVR"
    "evS8Kreen4Jik60O6z2dFsDgM29b3P9FPR9AewU5sSvRQG0GDjDD3x5gnHpsX1WHa5K+zVwqE6SkXhF0UK/8hV1ng+fNaJ0H"
    "XMAR7buAWD3OhNYqdPDNMOX92WPK/QLEd7XJgZankP3RbJHAsiyC9Hi9u50DVjaeBdZjHZ+PULu7c3LUqiduiFQ/GCCSfyZS"
    "vDrs6bja5dtVXhU7UF+FMIZNKzQeD96Hp/BOMAl0y/oO39rlVwd0HDWvkdNBea3e0PUJuS63V8X+sJLHJtmphMeHAR5QhioH"
    "LMHVnGQToU0iGmjg2oxp3pR2g0auMQMOa0bzRrQDowQvZzf76WQBCzKfzOJhG41eJnMJ/3K00sSrAgRqUvFkYdXl6QvWUBsB"
    "WpzdFqCsAldDbnW3+skAocMzORFw+68y7wb+RfU9mntLUOd2OYpjUNN29z9eZrDd77wkxM+rclNsr0pNarcD6llWhwrdkzgA"
    "lkocOBvwxg1ueZoCZ+pR5SI0tQAbsKN9qwCHxy1O3HbnIZFE0GCwproJqLXZfRfRHZ1QaXuKHT9PyawJ/Dg+XoArk+DygpJs"
    "JyJ+MUKG26Zl6VbP3dKrfjJAsfgMdu03W5DDYJCXq/wq8z5ldyClctyFPwhuho32OUMleomKdwBwFzHu31kTE9J2b/M6zjSg"
    "bqvbp3WSZBI4ukrw5FzQuenHFg8N+t3JZZLRkAt8DXcaqsmB3VEDjxsc+c3sgI3quhnCgz3vDWTaY3/ImQOTuYU2K4Ro4R//"
    "6SJgKbi83ntRvGB2+R8QRVxGWxwRWO26WKFUKYEiZAmCnliCvgkcBgwLC7PPFNZogKGNHFuDIfBYp/wJEMAJTdCscR7rsTL5"
    "eYDYwQsh9iSCRQgnwXxYi8N3SZOCPxx1OHpTzI/G1cS3UhifswqcM1SiaRcea8xJoK0xJx8MEDZ0Fdsu1LWKAk5sWwj8LZ5k"
    "G6TnHpQuD080YPQEPBMZnolf5+tdcbBERUEMLKW2tqhfrtU2eHFbLj/lrSn38ZCLM7OFRMvIyIl/3Mr54eMuB1cq7kNUtzxa"
    "EtiZCockQC7cge/Dd14uyFm9EzJN4nWZ/VB66Ge3sc770PitOh85oIIDxiDB8ZMVk4a53ruwVvTkZ8ZxmiD5Nt+DhJ3HQzKi"
    "RxVUZiEHaRDbPbJBzBz/Vk3QMivvdQ1iw90auMjx+IWQyaeVgLCVi4UO7worGTRnVwOdXhUQKGNoEtuTlCY95rkzXLLOLRD5"
    "JKVJbGQpTVyom5yIuieNuZGXLgSNtgm5Cd0Gg1y2GBtEpFaQJwO5INflFubJd8xZ/W07ptaWdnhS4ql5wIMSD1AZ9VJzWALa"
    "ZBFZW3bNHR5bg2ujIFIGiwGLtes0gK1x56Sbz8+D6pCXtMVoIhhvguaYEgJIVSLMkMHOANrDah32xboAQ8RLU/m9wQb7lg0+"
    "DJzRsNo0sZwT/EHgqUUMcrAYS6/YFpBNpFIfkXqPkIEgsKtsxE8CiQNzdjvCJNerDo0lvwayJX8SOJA/fdQj/cSe15kQcUn4"
    "IPGrToOSMmHkbzJO/tqmZQVwYgrgxEkAL85hs361XVX3P+3qfIhIiXvfeVfZbg+6dZA2cV011r0CsxUiyetcbFA+CwKngfdK"
    "oa1u6BBY18O7+zKSyo8Ai9uFCXPTF7YwtytM8pJaofHb0x7ipufDNDcCnQ80u57EXTpB98JskjpoX/guaUHwh6PyRW+K+UVm"
    "Z8wkkvK6lzNU8rB24fWkksZm/qjDtvb9x9zWY4j7ffkRt/SmAJHrT5LUa0Kgu4P3CabJtmX9bRx49sjnqviYVzkm4qLkRrSF"
    "iQ5gMDQCK8ypZNpYkPXlpdAFapMVZ2/mgzwkMO4zYd7flWrWOaH+XuD4JVDOt4ZH8blnC44+EA8SBl0MOKYx0Gh4B5878E7w"
    "QmyyZIb+NvbeiIvgrqdAB1dgJSk8HiOyjQlZh5mYtfWXBS6EeXx32RjtOs/2VZ30gPIIhuZbkZUExj9yM0hor1wertEs0VTq"
    "5lVpW6GpwZpV2oI3I+t5Udc1LapYGuu4UXxIkbFJbchQqefqoW8POOkg0QCxAtuA1krsIHWge/SCNOr3MXmMmD35HeQqFkIP"
    "wjcx2xWYaJnjr99hfnAdWMP4GWa35Eug2fawuf+pKpZqsEPCQUejP7d6M/25TtiHwsbZdKis+1KAbh2W4vMAleNzEdkQFv0P"
    "5VUJ29f0VItrUn5k+sQp3Ama7e/zGwoRwZ9/OBSf5fsdb26/eFcAOkn1fGeiSUKTpCh9ReJUS2YGIr5+H77BXRthJhJ5Q+yO"
    "l16X+FGIiKxmOwq8P6bPIT7EXpvD2i1iCi8CF62B25emWBF8Fjzh5cHEu/+zhxmEVh74Hbi4yg145HD3tm+TmUx/0vUf+jNS"
    "VX167kf4YppYUx7hsU7lUaBIvzeAcFSVkBo6ys8DhAyehpAjImcOqhr4mGF5/AknGP6uQRgPY7KkdgXEzrxPh8uCkgvgxiYs"
    "fiLmwRN4ns4Uwn6D9wx2MAseQEIpwe0iA9mN/YXGHl0Jg/9YL7NMUp34RyKGkxyDEnvhReDV3niZpA6sEp6aVR4aYwW9MTvs"
    "y839LzfF2rspIGME7BxMwoG5yiXKX/Rcok20Rzc27NfOjVO87DJw2zSvQHaBSAroZdJIZ2BkbgE0p2V+XW5vcqkgoMu8gGMc"
    "blKICS6sM2GGC/A1faFw4u/qS6PwffurgDnknF/ij8Jjg+Iufmr3KgFHsKm2D8UVZxuPJSubuqi2kimdObCpc/bew3nVUTx9"
    "l68qeY8mw+ynxh25RPe6SMG0eCE/ZZfIeWIcKUKwWujYBE5O5TDezLnO93BLCm/0UDq5vCNzk8GvwxzLJWqmkGXWrIGm0xCO"
    "QC+0PBcJkwpIX9h8lccAF9dsO2A5NjFgNyxCzx2YJH4+/eXoy9G0Mk4Sr/9+dL3Cfk+KZ5/Y4q5Ia/P25Xh2UjwdCJa8NIJB"
    "qjKsRzhzIph8mfR3H45aTAdaFaJeSJ1ho+RZ08TiZTsFI7+XgsPgZLK1BoglqYTWkjTyXUg6fxJ9YoQTKltvsuX9T1txxxRM"
    "01hm796Jkxo/4NWTHPX19f2P18UyN6K78mFz7/aDC/XhnJR7CY92AZISw9QZ6ZsVqTfa/UnFfLBfnITk2XqSPp4YjYS4TKmB"
    "58OJOg5KWNGFUdJHkehP4b8KhKUSTSKnY2AHlzKkBSCOUzL6fH92lG3i+xMR/vPttzH6uKEHk1HGiIIDf1vDuKLhwBGL59Xx"
    "bMzwrYggUcIEHqObwy6HxESx5m20mfaUvFeq6nzXkAK9hZzL+sqp68FxjSdNSbPqO9hyz5awAFomGE/gpD99YdPqjgBNib8d"
    "oGwYw7frdLGbiuB04/a8dLqZNPPdvFLN66Rjib/JNST+DrQUEnHR2ae9F9glAz7v9U0NAhRpJSYoPm6RGHpC4HL8+/7Z6QmH"
    "VbGHt+qjbw4rum7VBPwgv/HBTivW8J12tUc8crnY0yeiBUyRQN7MWQto/FJ1VBLtUkEhhhfkFO46wjAC5L7UQPfwhgJf5RAH"
    "BnlmH+axlZSEYE041YBeyP90KGA1ciFTlgeIOpVKbZNVcVMIxT2JZAJDpJVTqmFQFpA94Ugn+NFQRTklHR6fcmRkGjkQ+Xm9"
    "j8xJcHlAR8ayxMwc6SwEKav5EGk34A3WIS8jngUQvaHLrq7HgSyHh86hIquswti8ppuKc1yocwvmou6ix194PBJCHdDB81rh"
    "oscROFiCg1gmeonnAtYmQeEaO4aq6tfRBJN/i1p19Ld62TNJqfCNPyeNPGUyENN41q8VDEKkzJUuLDZuFRseoTR2UvfiMzwQ"
    "tHv7OdxcuizXOyEVM/CAfhb1O+iAKK8qEQSq7WnjeKi/l3f3j1AQJGy5G1GvZpwIi9Y1HzBXRQIda3clYQAJnMgEz17y7eLQ"
    "3vMdvEICQsstbI3SbWfnlvBEAYG/yypMYxQ6IhSFBQd4tsnXlO0EohdSAPBCHGaYVGtpMlL+pxS2iQfFDa6MLKh2DlkhDUIH"
    "tMz0Lh8mICgoe66p8pcAJPzzQs6rgQHA9f7/kilnjXcqv6QTEXCAQ05HBQJ7Yuhg2oPDgQOC03HAY18ABi0KloK3DhstrMfd"
    "30xCIThrjDowTnt+WpLf+oRshDnQA8yBA2nCxybNSZMQoHJBgisxc/TtZZtLuZV9mdINOx2KHVP1Flk7WuTzwy8sKwRA80ea"
    "t1c8Qsm7sDp7F30n8/EoCF+vCpx19S50B+/CgfDRI5/K41IKQJvPr0CLLSvwyMENKEooyG+Xh2qHlBcLB4XJ4HeDH4DKfU+8"
    "r/GkRWtAnOnL7DoTMXB6GxI2tbeVYxvG5HTLbotR0QIv2dEQwGgLVwQrJA98KldsrkGzYTEiixSicmnhJAq9DJbjFsi6yrAo"
    "Zq5w0u/p1jUQD4TxYQcw8EeC83ZJ3E2C+7BTTL4qp0oNxEafMgz0gz5TXEKFjkj8BGvFPvEVc74Mo4sDT48oW/Gvxbat9efA"
    "vfF56BQuSQaivkCO2aiQnlqJ1zrXoNCZI5ILGqb8MCiCMB9x316iqEP+YqK1mVswEzJkJmIDTBS6xoKVTwMwm5QCExofm1ZB"
    "KhFqBy5InkOvOP6aIm28+CGaRTsLltK0V+ycjVItzBnZjWoUWHchz9w1YHTqncpWimrkDLqBUbYqEkLULa6NgJzKrUZ1UVOp"
    "OGtSxBiLtyBgCO18GjW4f5v6yc1mUmw4AXicHqKixHGBC9AxmocJc5wWkj6X+nnsLcaYNgtbQ4LX/bDKujwAlBBB3Ow9e7En"
    "V81SmZ38/+q8LO10cjlQa3FOG/p7YN4NaHyw7QLwgl1l67Uo1AgeXQjpZrAlFZNXk7TNu/Cfm0xsWTngYZu2hqr1LRKSEK1r"
    "314aIm4QOmrTNkBF0yINHCt1dZit2DUcOvvqYHKBU8j3TKTuIqKNMH+IzK3nwAVlWmOYt4oHZK4xI0sjs/dF6OBt85/D28b5"
    "ZT9iMXywBuT9/zCeKdXwtx6UDwTFGAwJVCohZxV3IhT6oAxy3JIbvSa+nOWa7uPsXDaquLzP7xwCr9ZxiWvFFU/OOVeOcz5j"
    "sjWOAE4lXSxgWR/PzJ6vMR8u2knsEbwsgeuLdUkng2Y4+icxfTCoU523NBguqZeXYNZrLh0tkyq/zZZAqBwvENr98H6nZNMo"
    "uJ3cLQMiH543Sjb5Tv529zvGj+HeYSS1cLCjgnyToUMhjZQ6Pj8crrA9AoQt9nmdz4CNusxKPvBN8woo6iC+tmw5H20r3pRr"
    "7FCibUZsIdc4jOroiJxe9eWS5zUUgRJ725SowarnNBiNAvl9deBsjEbHoI3POHhS/OhlHQhRmtCJsGDblRUb+DEfpY5MvlYs"
    "ziBzrJHNSG+mmWgiLdIiHmHWnL2XgtlJYTQwEW5RwbCZeUZzBRdixs/k1GUyNWCOjyCpqjq+9rGEYiEVZbrITiG6BY5Fmin3"
    "pRIRyzhkXbj1HBgkBzVWFE2j+bfDLluYt453EUxY/WJ1KLXA/fflmjJJqQTb2gZRbXcEk6TNG3e1OyDWfh2m94pX7GE8OZyV"
    "NDzWVM35MfHlg4IK0q4hQUDILSjcnCcwFwy6BKgrccQZfBs9XYDQp8YX6RyLRvSUhC5hElmooPhM3E8pytBnJF9+ypRpsBy3"
    "mkRa167AKtwBAaFr5aH9anu3FPRIwDieBclfZQ+Ny+uhA8WDp6X4SeOOGCD/s0dl0cGlPWFrGqipLxgpEYEpqMUO8EHIKxNh"
    "IHfWLQrfXG8PKIXH7uJNTR/vEZCVAvBdmOx2N7zA6ZAbmAjvrI+e435/HRLR0zdtbYDeDS9bJVLeDSSc/IQnaYEJXGIaFPmh"
    "QnbNoyeO+D+Tz1WcCl1l0w/YLe8EGkf3AmWVTF/PLpCfB2gfvdBjfiG0TCzXxUZsqHQI6lLW/dbM8CUdGbYT3qcacuSltffy"
    "MRv5jIKoH+06LPaGr25wDjXvIRLHL1WSR8j2r0nkxW+O39liAir+wO5rAQuF7sKaw7VIR+9qBaxlT2sAWWrrBSkWLnpb8pIP"
    "c1Hsy5+LfqTHa2/tPBBfCWKbGA+Dpulp5Ef21orRaO3NBKxJcQMk32QxMlorRg50n79g7e019nsHETiZkxQPhwlvbrZmAlzb"
    "Scxv9EjcPaVb3iHjTQwnMUv4IcCWrW6C5MMJpjsxnLic7+kLpryPq5eIo7Cnpp1Ff4awoDx5L+oJgAKGpTZobCH4CLM2k5no"
    "OiVLz0VWKRClg2o9h5YU9g9BiD0vdCkRuUiJxTNIidMeGRGWCVy8ER6Xt3SVeuyhIaagW/pabaHGoRMmtUOneSFURIiA21PY"
    "zvkUUTERlYOsOGjSRIXOOwJGFbFD1nCKHp8rV5Ch9Br3UPyGmjCMZwoaLI5qVogQGD+sE31qxmiVefDQvBfXuQKmB6gzayj4"
    "WGSLholuUGg48C1BjX6gDgziv2RNQyTqC2/Bax+33dzBtJAJKApZ5Dzk68VMNZxIXgBhmCatKwHZSyImvLI5ArqFRVS4bAl0"
    "XeNMHBRO/9HchQ/J95XF7y7vf90pyeJ4iMLKZj9Qhof3DdWelQ0YKbEcw3pQ2wV97k1Xd6yefolJEFpYo5lGySjEJFsJ1KW+"
    "Mvr3CRod7u+JNkFq6z8pXyY8LjAlXBwFoT0VLWyR6ym3zEDX+k3ycNlUNQN4m7Xm4Hl2usX8DKwkaxu1zW3U+kbfF1pfm0Vb"
    "DujOWx8AIfw6u6pyrGACKhxF4CDE89u/LTt1kTDZFdO0M1kWGcG6cFIzAdwBWFF8IJJ6hL2fKeY5ERoXtaucOojaG5ky9ZHc"
    "ETBZygqa711qr40UuKgw0ZPy08i6z1gFB7q5Yp7JLqfgH2ahhFhQAMyArTAOoli/qXDYYlfroO5EhYHZWecNNRcLr4dkV5CN"
    "NVBL4+saqtQlItkcJYy18krfElyse4mFDOSdgWDWvhN22ylTftXuQswlwhc0LoiZS9b0hS1tqxdJfJVHz2TCYcR4VcneEYae"
    "OzBlfB5q05F51QEqHK8joVkf56OJpBhZ2I0tAWKhKlGhXYkKe1J93E2uBhvD4FLx4NSpcCjfZ1Saz3W5d4u3w4tUBn33R/ha"
    "nowGY8RP6Kz97V+xwMFv/8qwwtfZenlYN02vG+TFSNxuWD6X/lzYPLTkLSWj2Err0KD1KHCaJa0CYhUcnaShC0mDF0fS4Ldf"
    "+sKo2hI3KyyGDds+dFPFGjQNgx5SWuBw21JCYGmoB0jDwIGG4Yuj4V//5X9hWfy3bE4ueC2r7P7nz6RxkO1DDScike+K6duo"
    "/PtBW+Usimz7E757K8wHuzTv5kCMB6vtUxUge/TqCRDBzIHG0UukcRCg1Q9/RAu38xcXf4nVJ+Vg1NHhRKQZjOCoEpVWHRvd"
    "PJfeE3gQoBEAV7wYXHrLmCw2Im38VKQ9rZdqAWIKkwHeiGsob6Xv1/1EXpCcS317OKTjY/Tr1INYdXS3+dHoC5GbUNw6nEvX"
    "kwjf4F61V1x2O9cbbI0oSS+emtrmhCFfqtko1OzAWMlznhlHZmb/9gvX2hFv2VXU0vZwWRpF6NVbMdRinq5RoAJNEtdo/hLV"
    "chguuN3hP/Kjm9PzwYhQ9jaPwon8n8QB85fHAXD0whZqipQ6qgYwBBZ7wWt3olQVtbCw99lwUARqIBbVTp2eb5Bh9MRwIKBz"
    "Ae1HFRDHXrsiCZEcYXzR0C/lHaiE/tbin4pEFq33EuftOxaooQBo4E65VRcvmNLhX//5f08cy9yRExE7RsqTMax9FQtY9UXj"
    "uFBVvaBpepBaC6Kn/kBVO2eYdPlZhcZqeXrp89QfprBT0Pr8DmM/WYB2An+EyXgdHgejD1So1KGWpaCZaWGtAVmLUvgLZyXe"
    "ClG30BRY7FGt18LyFw7k9V8eeYO//vP/cd230Ne62C6bLSSaQhn/sLdjd45OleOgdu/G7k7sYjECyS/rdMZdQTGE2E0SC82n"
    "GSfLPNWE5eyj2oFpS0Ki2vOikIg11cQsfTCIDr4+iIi+61UUWM7QM02GSiUQZ4Qvb9ujbtMUB3S31y16WCcHLJKO8m5yupuV"
    "raldzOxsJrqeiO5Au0dwqj2o3IXSDjdXzzNsdi+sTnAvkuFJRglXHEq1UMWLPVYqVsLAG53U702r9mb4Vi2ekx7LeCRcUfFN"
    "d6uynhDTFeKgXvvP7mQ7+uyOZG21MeYwjoGl164MKbspEDP2VPBzsIZbGMZe1WbnK/aNqtQHdSPdwo+iwCQGpy3US54wSjWl"
    "DJIpe++LMiS25QbD5yV6ivA6tawzpV6ylnkBO7yZLTJUMCOMhqgmEgLDk/MdloUot0vzCEGVaGoXvlPzgtgpUCNLyhEpVhxP"
    "DXk8jR24JHh8LjltPvAU85H8KRcKU7u84ruZV0JDespSwo9g30xTq2sEnouJ0Rc9tbq94DHfUpYFprtETDBsKYqp7v+Snweo"
    "Gb4wasZTynmfhg7UhLc8LPB6KzJ9puSEmFoNZDEtfIlBqam1Hww85mnJgdJ0ZBMIWz5qqneBkZ8HKBm9MErOpz7ddmWPcHV9"
    "8bjHZjE/LaUMnFIp1Klvv7Qr58bv8Xyc2sUwPudJ2gNS36AWYPyJa8jl+sEAbeMXJJnDqWhsFbl6PhqxVvfCwCnx7lTYNMTR"
    "7nQpJZvqim2KOL3Aq1My9mfdx8lQiw4ndPTI4zAirCtb92Q78ELywvZ5NF2Qvsoe2WIZxXLrNcwN64ZMmoNX4B7dZXj/a7oQ"
    "lzjCafBG4ZC/vwRDg6LHfjD103dCnWrKQAnNCBGypaxMQ1txn4ejh7M6I8amtkz1qi7y8wDLzE/JMg/Np78uPn9GUSmam1Bz"
    "rxTXTgBvS8FCcv1BKU8JGfigEEO5qfZdTKT/E/R916ver2sQdDlD9mvFsUX1gTXxxAhZhSlDKqRTzByg7GwCvJU6XlfYpKSg"
    "xaSghXY9EB6rqHWsP1fwuuAxAfNSpwNdEUAuWmL66Aw0rhMFpcB79Y0MyMi+//GqEBd2vgKe+ZR56/z+Z9hOYNJQ4w4aoBUp"
    "bIY0L39wcdeZrFYXQ8UiNFORFh1Es25pYdIdkKRR2nYY59w+DXJ9Tj1HTJR6wwwOPY4gDRF3jxDxzOL55NSR8fRp0GtafKOJ"
    "d9yt97/S4geYVIybNhGT4G6ktJM+iyNUjIHYbnHEpsXxAAz0IhQGbN6kTAyT0kFBMeKwPbGcR9BYj4zTTUXEeuqmsyp+GOob"
    "orhysI4FTfVm+lpcU3lDRr1WPk4vSCl6fOVYklI2MbugAbg7p9aoHjzu02GPQ48YZAxi7LEx1eN/8vMA0/gv7bRAsYs13B15"
    "RjQDVpRDGC+UQ7Ab31A+s6ZeDJAiuqABtH8D+9nRyyNu6DjwhI4If5AExkHiwhPByQ+SUzQnqeueJjO1EOc78g/ANb7lvsBu"
    "B9tsLziF7UZiefehGkgg5X6iKiBJI/t9FBN+UKfSsZWS2QYlxyBCPU97UOipl8w1LTHvUhn9D4h3wufxmFgrazc7SF+1xZSt"
    "dQOCfnP/I3T8KhshLzrPedQ7lZYQnftWr0jrwNBOf7lhIW9gIQ5xFHdWW6UTdnRERveJjEGDt1yMK1dTB83Tj57NWuF6LEBl"
    "H7Tr4IY0uflh3fbZZ4ryZZ/vfwXnQX2X/JoaG2XkSkK4cmuJ8e3F8awep/bHc2qEq+xTEgIBmQvRm3pHJtOIk0Io/el21RR6"
    "A2NzUmiMWL0jI51+zZ2HK5LZeSqof3SfVHJBriOTXNHieUzDTeE1FxkVvzhldw7KLnqrw+Mip5V6919RNS+oelckmyR1PXHh"
    "NBTuLixGDesErO9Zs039UdFSF3R0z5sdEVab1XXZoUyIyoyi96ayVUusBP6DlSXmJ3K5kdaCLiUoO77OqMxEFMz0ku1w9mAu"
    "YPyKWhAXy+KaqlAcsGxAtc1R3cEaKsVNqdWu0LSZeuCWTZohAMqim77T2USr0ohYwr7Hp5TqYK/TN4t5dcURJN0X6QDjEyti"
    "VjHxZ7EDg7hdCX4Qb4zt64rFKOj3iFoUUDLiQI0BVgfsgYIrlSpdgaC4SVl3X6cvF4nWLPMf0Cfb/gLqWQe9IzH1xysv94cb"
    "N1dbPeZCzv9eIoGXemYza+qNOqwNyixLqPVwIUZRzSQ5DVBbrV2hBv8yBf8LepGCP6JExOyV/a7rqz5teejX6MrT+N9BKZsj"
    "fgF/efbVyNuzlesN6ac47lDbQr8weP7Lj6g2ISev6PTDkijoC4ZfSH1/cjoYQYcodqicoUhYCorO407z1zpp8Bp7yJeC5sNc"
    "TBFBCftCTiuOGSJ8rLNxU1ZHHYbpfgJoPcF7MZDqQzLNrxKu+dVI3NR4w0is2IAn0xsrGe6NRawWnRGroVy8/wV077ItEOWn"
    "M6VrUgZ6ilEnii7MgFhCVsSv9XJQYqSoK+bEX2IqBHtBsImEIUqxJLTxFhzdEpFVKUbAFTIKBTAVLP25xKqPk3qx0LioFz4r"
    "dDUkWnk7VMySOCY+K3lb62NbedZkkOipO5Kg/itsrY+iF9cGMAHrGx1FnGzVnUpioDbOpUc1tdNtRQ2Vi6JStJgWLoRMpGpn"
    "faImiltRE8VWYQmPe5pXuyEzJPFMNNjaubEu8SIXJS45KVM9zOaTWxGXyp/RPkzjGe9ZwncUnXgl8xvEcmsJVTAN+bip5YhV"
    "/07jjso9ND31TNMnZm8B6vmpqQth5mdKGHFABrM+wgwZK1pFv6CW19QuJrRnuM265HEAQjFjc3rWf2JktLkI8fT5jmS+C5ow"
    "duSxrHZBFUcrluve0I8XpSbAxQILSEuHOubf/OevSKT7i1cP1hlhrguJgSDBgjqCzCy9iOTrqj4mBr6lARiXWTCl9uiLYSWR"
    "R0atq+eIBhsgWtgL69FzB4ZaPKPTheGoViyjk+q6FObysqxgfUAVIr9cRK5HOAxh0a4OmJfCyHg4g0DMwTr32s96S+NmjClI"
    "RV5iLARpGjlK9TRqxWkavbL3vHrV12PZGaEhyW6iwrfCemW0wuoNHCEnOd4ifxr50WUFCA/A78QPqA5RFHK4k6JQnBKxvOHM"
    "kd6h4jAI7S6PcPaKbYRoAh2iqQmOFzevRssbx7vjTyZvQAwDwTLoCoUkBOGRY+Ifpn/CUsIhIW5/wVKAcZBYXBEfD7R7jCMC"
    "swT3pZOAke9eyOnfS3DCdu9PtaaFh8QBcPbWngahlte+KD929Kn5in7u280EP37VJ5r6fwaTou3yA4aY1USdtVsN08J30GD9"
    "4Dw1WGnh92qwQ7rlLNZuwNYqxYwcBYwnq6PBOgAR113N6XmXlOGJciBSeHYa7B6vMFNYXqqwVNK49dkL91THG4pnbadmNPG7"
    "d//TaBenhCncP3SUJw4+zmbce/G+6HA54/pqjnRtMjixvk0Wm56Om1bllZ478FJ0dry0OeDxeF00Odbk2v2UXWLv7H0uSu0B"
    "j1yuZcsjcRKTIZS+0lKelEEQgV1V9z+Ok1BB65ieibqfnBkkh3iZKHlQ41ZP8darS3zPGdaiLxSEnUSPFT/DMhqHGV8Ov4ue"
    "UhPfjdfi0xtKD8+2VP0kgfBgRXGPnySY9YoAf6JdzJd+QBHXcGsZ7wqEQsfG9CdqFZ8vDyNyCeDtjM0uSU8U8ICLKngnXeqs"
    "daEJ6j1TX9xBjTXQ/R+YoSReZTb+d2tk5zpOKn4J7Y5bTMePLS3dxJnt4/3MbHNJ46jzxC29S6GonqYWOqmdoGuywgEuv4VH"
    "NbIgHnBRB53Jf7rkImWlwtu2ulufU0JZszoLUKGrmARV8ZArcoOvLDgPg212fNecl6NNM7lriRuiTnjyHfooJApu6xZiPR1V"
    "BXt3uDu4rYv7cmahpN6co9mcaR/DgrKZbg2QPgLq+8tle0Vnv71uvakoZRs43TRQ9wEKqsjYYyL7OmIbXNJhlvalaXYh1PtM"
    "m7uPTiP7VRKl4sen1LFXRBRqxbeyoVbIkatWDFOT+2NamT7/SyzIw9LNINsgJNs+a2D00W9UFXaiXvIiqBe9RvIFb2SDgdEC"
    "LcLFe0/JyrbuWyMPSmdIqr3xSOfk/EXQb45yjlphkvrGCbuvrqrDdVPf9FYYZk0zXprmvVBrfOygac9ojuh7SN2my1RgKmdo"
    "49HOYs9Vnb6j0dAzmfsQ6D189bPXgfjp09ghI33ozQ0JxRoBQxtWJlENEigZbgRkj7RHAiRHUtc4N/d2ItWXILK2T9CUGN9+"
    "rzNwNE1URNStr6DQo0H5PXc3jaubQW+8jFhjccpz4cFc8e2h2EmfFSiX93+B3w5ZUKKVJVYGuJNLUbeBQ3sWGu/54l3NcSVG"
    "i7lgKMMskNr3lUw5zXEH46B3noIGKd4Ku7REFSbm61vRTw2OLL+9v9Iq7KFv6S7sYROVVZe8gvp48/u2bl9saW0vsDRUBbff"
    "gYOO+AWa/eyKO8ukyg9oWTUcPsacYrvPL8NQ38ca2kHTIKfHWgu6vogpCcCY69VA0iKUR0HMSSo/Zmy2HoCdvg0GqD6Z5Mej"
    "AmVETv9MyQkSBlrVQP5lRvvjjm6UEF+/pw+RKY9ESBIE2e2HfpMB1QRSAZtf5cnrggXej4KrlmPMde81HodUYSbH0AdeNkD8"
    "wjeO+ukxaPWZ9j0InVSZ9YPz55ygYR2stY9/kEZ7BOvs9sX+IBsk5Vrba9NBRCc5Xf/3VZW3UVMvpFpNLyhFR8NGp+H0HnkI"
    "RByF4zeOKvPRPwencf4hNiZ1+AknVbqdQszP65lK6IfRSkZDgiq6fTq/Ig/r6RyLfvToirEr9f4T3pqle05ZU19NJgls1bt7"
    "TZ9oNSFAJhNQf2kxB68JKy/V427BOm3NGNpfi4EkI7nRwoAJ09IXtgyAkRj0bHIBm29wYY/303MHzojPfmOjt4HUcWZXw39p"
    "NW8x4bfEzj+4z6g7OWUA43lcimlQi701XNCuG3o0mNoPfcK9fJWXI+Kw8PYm3+OHLtUWp7t3BqoFXA4qPUol3l5B7LyCQv+U"
    "nuUhfHH5DJ5eldT+sH0GV0o/lZoSAYkc9z9Wecbf26avpZ8FIcLBACCpVqUAAT5/Ma/Z3iClzWbtV5HUSNW9DLqXt93htu0N"
    "JES2h4UdbBvySxy4wUV7fJmMAGfJ/c/0Ds8N7Tt20hAZXguQbwWYN1olffxatPElNzsV95jVkY/uTbGZRLrDHkchIurm96DA"
    "3hJT8Wj1uJkDv4SPzS9jrx+CI6+QC/WODDf5YUe8s8X6kFe0N7BR31z+rSoFxGKYvw91A/KqHv6BLyjSACBUC6rjIYjWoiKd"
    "cXglVEJUjhO6J/o6ok6uc1Lp6YlPGReJ1Y2RxHKebkGR49ChU2YIETbJVMWmzTWNHfgneo7zhssybdenTijNPOQzSg6eeZgz"
    "Jnclcg9+m633h4rO8UT7+gFSSMECxr4m+CgMCNIb0c6lZZ7XPiXoJW/qvilJ00GFuzan/w43EeWCFE7Qjw5/76EHJ+fbEMRP"
    "8bnwk3FNzjzPmy5Hy8OdVP3o4FloKzFCUqltk+gKEc5pNi5e1E24umyh8++wQLLDa/oTS0gsxbvgWjq76CnJk2itRyZ6qjtn"
    "ef9ztcR9Ax/hVUg7jnVmh8JpWHLy//1PqufEOhu+YaaF5YehsOI0OXTpa0mOs+E3oqtd/YmqAM3TiVWu4PNjzgceHRw2hAjb"
    "LKQfm9bMgfd6o3nEMvPnUlX6uAX/V26vwO23EksJtn0BS7n9iCUS0P/W8M38KL75Vgew7sxfa5MK2UyVtqYYtnuIwond6qEv"
    "eg6RoxBpVNouCqwya8Wj1Wrxa4cDJj1PQaJtOkaGNPaRm8KhnuQ0SVd4ODX0G2f0slDHtfPrN3kHq7kTqRfP5gHhLuwjeIvd"
    "G9e/FmOPc09XNSmwb66GUGRP7P6I643YU7ju1A4QDWZf/bp+fpg52CROofQnExRGSKxcU8OMeg9d0uYHO6QqkADEJpGjCmmZ"
    "C4CImX77xVQdozr7NbHu/yQyBPX9r5fA7l26D0BtFEgVHmt39gBtzVCHI8Ap3P5kJP+dasVjRdDdAa5Q0yU0sOeF6Skbu9Z+"
    "DbLReV9GyXu9dGA6oEGnxREuilHwWq/EI/og/OCciE9+iOyaqqoILwQa2zvpgrgz/AwdN0R0OjeERALtfWHwb7I7kDtvhUsE"
    "kxErYea/aV0BqmfiNfXNTpR3cDun9R9cV/fj3BNHIUsei2E0+ViZk9sicGDB8Fy0ze9zj1J0oV4k+D+zilbX4oinek8dNzxd"
    "How4T7yqoawkALXI95bvYSXepQIzVG4mpyBZ45O4cPPMBzMK5zd+8ZDki/XGQZJ6jBFzHCqtb55Dgj3YUptzPkkduCo6u4Ot"
    "8SKqKp845hD2XnAGZgXZD7JGqGG554Kabh/YqP13osfhPgcfuLja+V0BQOEHZpA20Q4XXgn4960X0d8LpBDmIGtFm79VQeJJ"
    "Aw22yYdhFX9xjyl8EsRwyh6U2Ku6vGnsIivj8/HXYzLED1LnNdxE9YH0DlipPoUotE67dd+JDUql1FVXhuj8bpNp52HHbJFi"
    "RjWcYtGZTBb8d3TPu+nRozDCuTq4OPvmbTr2kGsephxz4RsQWIE9AIm2f0QegaoA+O7Kwk7+7LTlwpXCZqIaGeV/Ns0GMNVI"
    "NGcQ+StgG02gM0M4gayVaILJ39Fk1i2RVxJ8JNXuYE7hkBW7hbwlAQovAE0wPIfA8F9RY3sS96QcqY0nAAM5GQ6i0kCUs+sH"
    "1jOMvuDTYl3wsiUiuWLEevhqtFqnXv1kgAWdcmNPyX0jKntflutd1tZajvEBLHJV/pDt8BzLPh/WVJ0kgPr01Ur2NgD1CRpz"
    "V5kMVsIQVEeg30dV115ei5PAU38LLju1Z80zmvcD26W6KSFDyXM4jtDCw+xtk8mvFVb91gBFmwYz8D5CL/gqu1xTmwssUS4e"
    "X5e7Ah9ejOtl7YqYFJMPQemEHa+JC8NH5MKTlcRSTy3qPyLulWF3XyrjfFffxOo5u+D/tDa4kpSS4uMZEUR1rRLHI6uZI47q"
    "Uge+VWn3mTvCA4g4nFF2FFg/pH6F2E8dmCI6W6ZQpBmsyqZcZVRcC06qHw6oj0OSeUz/mwNTLHgbjkaqmf14AtENth0Vfyz2"
    "ucITWAwNStBm1OcAPdk3aCVSfw7U72gcHnk7PPJu8s/vPCgv+DPNuW5h2VXzjk3nghoOOhVSrHI+Rh8ntomfnG2OdF7j78de"
    "VVuDe6J3mPLrA/v4eD8uCHsyrUBfRJcXLiIaj14JYnArc7nQTxrX+1QXRzVclcBLwnKtdtVqcr6wSpo9sBFb+jc54yQl0XHY"
    "8FUYx90tQ55JzkELsrHLV9DMOtt+Fl6nFQYFUWXOCxTXVbYb0mhKuj1OgyOGi/5jhh5UWEe4yidrfAhRT9TC/g1bciI2ikG/"
    "YuPbmxv7ZnPj8WAZtcXv6WjsGx2N/aGOxsQM82exyo7hCDxAt9A7xEmxzdZXGXFNDyMENopc0CgPq0VCdEW9C2+jk37Ad6nl"
    "1uPYFRm6D38cGifraExMkz71CTK2KSV5hmR9xxms+/JTtqMkcTySyaSqraddRjXuM/HSWOMJp+I1niONjnSq1fgb7lKMA6gX"
    "ulVqweOOwvMQzNw6Jys4sRXjprrwkp8HeG/x4qyn2lUCPpLZO3KO3IEOOJs4e35gnPSqYJ2CmbjHOXDNDN+oTZVoYr9+MOja"
    "YQAzt8tMkKz/ZtR1UaC5kZHxCELqZBYS0L4CD2ypK7loM0NZ0YAoz3uZxUhVMRRRTFH4pX2I0UytWwoEyGO25IHf7WMzCpDo"
    "mNKCOKEq6lbm4LkECcRCSYRsMfIOKQo/5LiJZSNWLAuANV1RYoCPyJQmI+RIW7SHRApYlz0mUOOGQSNUQ4FEG/AZGFRQNgkd"
    "ND40LBMmiHm892u18RTXkURcc8wLy9VVoTkhpowi3I/jSbUdP3jpNrZca1SAuHADLH5UW6qG10KzV023IGqr0nrV2Op11CZV"
    "sCFSQ/6cBgdK6jChn8zh4ofnajxj9ApSN66hFDdIb+i0gYEp8KoA7YHwkFJVol6XbykGdf/rWnR/WBbbZSk6HK9sbTiUpRYT"
    "YIqHfP3DsPuXxmBKhPDJ+6hNwP9nZtCgU6ixQQmXiDro0JVtm3orgPCe334crGUiGeisItui0CqzDrwUnbP0e6Az5vd/r/pj"
    "Ei77UOxgq28EfMqgh5JcAE5MhmSW1RMTm56YMQBZ0cP7YGLDBxO7+GD8+FzPlG5DT8guAYkFS4QcADmZbTdPWLKP97/uIQrX"
    "BCbXMDdNIIeM0YrkxJky74MNbaXfZjzVcnsG7NlYjsBUCbuRHR5vZJtYOdjXBj58gDI24pOMhPv//4ObqfNRAQA="
)

if not RUTA_DATOS.exists():
    RUTA_DATOS.parent.mkdir(parents=True, exist_ok=True)
    RUTA_DATOS.write_bytes(gzip.decompress(base64.b64decode(_BLOB)))
    print(f"Corpus escrito en {RUTA_DATOS} (copia embebida).")
else:
    print(f"Se usará el corpus existente en {RUTA_DATOS}.")

sha_real = hashlib.sha256(RUTA_DATOS.read_bytes()).hexdigest()
print("SHA-256:", sha_real[:16], "…")
print("Coincide con la versión embebida:", sha_real == SHA_ESPERADO)

### 4.2 · Carga y particiones

Las particiones vienen **fijadas en el archivo** (campo `split`), no se
calculan aquí. Es una decisión deliberada: si cada notebook hiciera su propio
`train_test_split`, cuatro arquitecturas estarían evaluándose sobre conjuntos
distintos y las métricas no serían comparables entre sí.

La partición es estratificada por categoría (12 entrenamiento + 3 validación
por clase), generada con semilla 42.

In [ ]:
import json
from collections import Counter

registros = [json.loads(l) for l in RUTA_DATOS.read_text(encoding="utf-8").splitlines()]

train = [r for r in registros if r["split"] == "train"]
val   = [r for r in registros if r["split"] == "validation"]
demo  = [r for r in registros if r["es_demo"]]

CATEGORIAS = [
    "suma", "resta", "multiplicacion", "division", "operaciones_combinadas",
    "potencias_raices", "fracciones", "porcentajes", "ecuaciones",
    "geometria", "estadistica_probabilidad",
]
CAT2ID = {c: i for i, c in enumerate(CATEGORIAS)}
ID2CAT = {i: c for c, i in CAT2ID.items()}

print(f"Total: {len(registros)}  |  train: {len(train)}  |  validación: {len(val)}")
print(f"Ejemplos de demostración (todos en validación): {[d['id'] for d in demo]}")
print()
print("Distribución por categoría (train / val):")
ctr, cva = Counter(r["categoria"] for r in train), Counter(r["categoria"] for r in val)
for c in CATEGORIAS:
    print(f"  {c:26s} {ctr[c]:3d} / {cva[c]:2d}")
print()
print("Ejemplo completo:")
print(json.dumps(train[0], ensure_ascii=False, indent=2))

Un ejemplo del corpus se ve así:

```
entrada : "María tiene 48 caramelos y quiere repartirlos por igual entre 6
           amigos. ¿Cuántos caramelos recibirá cada amigo?"
salida  : "Paso 1: Repartir en partes iguales es dividir.
           Paso 2: 48 ÷ 6 = 8.
           Respuesta final: 8 caramelos"
valor   : "8"
```

El campo `valor` es la clave de toda la evaluación automática: es la respuesta
en forma canónica (sin unidades ni texto). Comparar `valor` contra lo que el
modelo escribe después de `Respuesta final:` nos da una métrica objetiva de
**si el modelo resolvió bien el problema**, independiente de cómo lo redactó.

### Métricas de generación

Antes de tocar el modelo definimos **cómo vamos a medirlo**. Fijar las métricas
antes de ver resultados evita el sesgo de elegir después la métrica que mejor
nos deja.

| Métrica | Qué mide | Por qué está |
|---|---|---|
| **Exactitud de la respuesta** | ¿El número final es correcto? | Es la única métrica que responde "¿sirve como tutor?". Un procedimiento bonito con resultado equivocado es un fracaso. |
| **Formato válido** | ¿Usó `Paso N:` y `Respuesta final:`? | Separa *aprender a resolver* de *aprender a formatear*. El fine-tuning con pocos datos suele mejorar mucho lo segundo y poco lo primero; sin esta métrica confundiríamos ambos efectos. |
| **ROUGE-L** | Solapamiento de la explicación con la de referencia | Aproxima si el procedimiento se parece al esperado. Es una métrica débil (premia coincidencia léxica, no razonamiento) y así hay que leerla. |
| **Longitud media** | Palabras generadas | Detecta el modo de fallo típico del baseline: divagar o repetirse hasta agotar `max_new_tokens`. |

Nota metodológica importante: para extraer la respuesta del modelo usamos el
marcador `Respuesta final:` y, si no aparece, **el último número del texto**.
Sin ese respaldo estaríamos castigando al baseline por desconocer un formato
que todavía no le hemos enseñado, y la mejora del fine-tuning se vería
artificialmente enorme.

In [ ]:
import re
import unicodedata
from fractions import Fraction

MARCADOR_RESPUESTA = "Respuesta final:"

_PAT_RESPUESTA = re.compile(r"Respuesta\s+final\s*:\s*(.+)", re.IGNORECASE)
_PAT_PASO1     = re.compile(r"Paso\s*1\s*:", re.IGNORECASE)
_PAT_CATEGORIA = re.compile(r"Categor[ií]a\s*:\s*([a-zA-Z_]+)", re.IGNORECASE)
# La alternativa de fracción va primero: en "3/5" queremos capturar la fracción
# completa, no el "3" suelto.
_PAT_VALOR = re.compile(r"-?\d+(?:\.\d+)?\s*/\s*-?\d+(?:\.\d+)?|-?\d+(?:\.\d+)?")


def _sin_miles(texto):
    """Quita la coma como separador de miles. El corpus usa el punto como
    separador decimal y nunca la coma, así que la conversión no es ambigua."""
    return texto.replace(",", "")


def a_float(valor):
    if valor is None:
        return None
    try:
        return float(Fraction(valor)) if "/" in valor else float(valor)
    except (ValueError, ZeroDivisionError):
        return None


def valor_predicho(texto):
    """Extrae la respuesta del modelo en forma canónica.

    Prioridad 1: el número que sigue al marcador 'Respuesta final:'.
    Prioridad 2: el último número del texto.

    El segundo caso importa para que la comparación sea JUSTA: un modelo sin
    fine-tuning no conoce nuestro formato, y penalizarlo por eso mediría
    obediencia al formato, no capacidad matemática. Con el fallback medimos lo
    segundo; el apego al formato se mide aparte con `formato_valido`.
    """
    m = _PAT_RESPUESTA.search(texto)
    if m:
        linea = m.group(1).splitlines()[0]
        v = _PAT_VALOR.search(_sin_miles(linea))
        if v:
            return v.group(0).replace(" ", "")
    todos = _PAT_VALOR.findall(_sin_miles(texto))
    return todos[-1].replace(" ", "") if todos else None


def respuesta_correcta(generado, valor_oro, tol=1e-6):
    a = a_float(valor_predicho(generado))
    b = a_float(valor_oro)
    if a is None or b is None:
        return False
    return abs(a - b) <= tol * max(1.0, abs(b))


def formato_valido(texto):
    """¿El modelo produjo la estructura que le enseñamos?"""
    return bool(_PAT_PASO1.search(texto)) and bool(_PAT_RESPUESTA.search(texto))


def categoria_predicha(texto):
    m = _PAT_CATEGORIA.search(texto)
    return m.group(1).lower() if m else None


def _tokens(texto):
    t = unicodedata.normalize("NFKD", texto.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.findall(r"[a-z0-9]+|[^\sa-z0-9]", t)


def _lcs(a, b):
    """Longitud de la subsecuencia común más larga (programación dinámica)."""
    previa = [0] * (len(b) + 1)
    for x in a:
        actual = [0]
        for j, y in enumerate(b):
            actual.append(previa[j] + 1 if x == y else max(previa[j + 1], actual[j]))
        previa = actual
    return previa[-1]


def rouge_l(generado, referencia):
    """ROUGE-L (F1 sobre la subsecuencia común más larga).

    Se implementa a mano en lugar de usar `evaluate` para que el notebook no
    dependa de descargas en tiempo de ejecución y el número sea exactamente
    reproducible.
    """
    p, r = _tokens(generado), _tokens(referencia)
    if not p or not r:
        return 0.0
    l = _lcs(p, r)
    if l == 0:
        return 0.0
    prec, rec = l / len(p), l / len(r)
    return 2 * prec * rec / (prec + rec)


def evaluar_generacion(generados, registros):
    """Métricas de generación sobre un conjunto de ejemplos.

    exactitud       : ¿la respuesta final es numéricamente correcta?  <- la que importa
    formato_valido  : ¿respetó la estructura Paso N / Respuesta final?
    rouge_l         : ¿se parece el procedimiento al de referencia?
    long_media      : longitud media en palabras (detecta divagación)
    """
    assert len(generados) == len(registros)
    n = len(generados)
    correctas = [respuesta_correcta(g, r["valor"]) for g, r in zip(generados, registros)]
    formatos  = [formato_valido(g) for g in generados]
    rouges    = [rouge_l(g, r["salida"]) for g, r in zip(generados, registros)]
    return {
        "n": n,
        "exactitud": sum(correctas) / n,
        "formato_valido": sum(formatos) / n,
        "rouge_l": sum(rouges) / n,
        "long_media": sum(len(g.split()) for g in generados) / n,
        "_correctas": correctas,
    }


def tabla_metricas(antes, despues, titulo="Baseline vs Fine-tuned"):
    """Imprime la comparación en el formato que usaremos en el informe."""
    filas = [
        ("Exactitud de la respuesta", "exactitud", "{:.1%}"),
        ("Formato válido",            "formato_valido", "{:.1%}"),
        ("ROUGE-L del procedimiento", "rouge_l", "{:.3f}"),
        ("Longitud media (palabras)", "long_media", "{:.1f}"),
    ]
    ancho = 30
    print(titulo)
    print("=" * 68)
    print(f"{'Métrica':{ancho}s} {'Baseline':>12s} {'Fine-tuned':>12s} {'Δ':>10s}")
    print("-" * 68)
    for etiqueta, clave, fmt in filas:
        a, d = antes[clave], despues[clave]
        print(f"{etiqueta:{ancho}s} {fmt.format(a):>12s} {fmt.format(d):>12s} "
              f"{d - a:>+10.3f}")
    print("=" * 68)

In [ ]:
import json
from pathlib import Path

DIR_RESULTADOS = Path("resultados")
DIR_RESULTADOS.mkdir(exist_ok=True)


def guardar_resultados(nombre, payload):
    """Persiste las métricas para que el notebook de comparación las agregue.

    Sin este paso, comparar las cuatro arquitecturas obligaría a re-ejecutar
    todo en una sola sesión. Con él, cada notebook se ejecuta cuando se pueda y
    la comparación se hace al final leyendo los JSON.
    """
    limpio = {}
    for k, v in payload.items():
        if isinstance(v, dict):
            limpio[k] = {kk: vv for kk, vv in v.items() if not kk.startswith("_")}
        elif not k.startswith("_"):
            limpio[k] = v
    ruta = DIR_RESULTADOS / f"{nombre}.json"
    ruta.write_text(json.dumps(limpio, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Métricas guardadas en {ruta}")
    return ruta

## 5 · Preprocesamiento

### El formato objetivo

La salida que enseñamos al modelo añade una línea al formato de los notebooks
anteriores:

```
Categoria: porcentajes      <- clasificación implícita, la línea nueva
Paso 1: ...
Paso 2: ...
Respuesta final: 105 estudiantes
```

Ponerla **primero** no es arbitrario. En un decoder autoregresivo, todo lo que
se genera antes condiciona lo que viene después: al escribir primero la
categoría, el modelo se compromete con un tipo de problema y genera los pasos
condicionado por esa decisión. Es el mismo mecanismo del pipeline del notebook
3, pero interno al modelo y entrenado de punta a punta con una sola pérdida.

Como efecto secundario útil, la categoría queda visible en la salida y podemos
extraerla con una expresión regular para medir la accuracy de clasificación y
compararla directamente contra BETO.

In [ ]:
PREFIJO = "resuelve el problema de matematicas paso a paso: "


def construir_entrada(reg):
    return PREFIJO + reg["entrada"]


def construir_objetivo(reg):
    return f"Categoria: {reg['categoria']}\n{reg['salida']}"


print("ENTRADA:", construir_entrada(train[0]))
print()
print("OBJETIVO:")
print(construir_objetivo(train[0]))

## 6 · Tokenización

### Diagnóstico: el problema del vocabulario

Esta sección es la contribución técnica propia de este notebook. El tokenizador
de T5 es un SentencePiece de 32k entrenado sobre C4, un corpus en inglés.
Nuestro corpus está en español y usa notación matemática. Antes de entrenar
nada hay que saber cuánto de nuestro texto **el modelo literalmente no puede
representar**.

Un token `<unk>` no es una aproximación: es información destruida. Si `÷` se
convierte en `<unk>`, el modelo no puede distinguir "864 ÷ 12" de "864 × 12".
Ningún fine-tuning arregla eso, porque el dato nunca llega al modelo.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

caracteres = list("¿¡áéíóúüñÁÉÍÓÚÑ×÷√²³°πΔ≈—")
print(f"{'carácter':>10s}  {'tokens':40s}  {'¿se pierde?'}")
print("-" * 72)
perdidos = []
for ch in caracteres:
    piezas = tokenizer.tokenize(ch)
    ids = tokenizer(ch, add_special_tokens=False)["input_ids"]
    es_unk = tokenizer.unk_token_id in ids or piezas in ([], ["▁"], ["▁", "<unk>"])
    if es_unk:
        perdidos.append(ch)
    print(f"{ch:>10s}  {str(piezas):40s}  {'SÍ  <-- se pierde' if es_unk else 'no'}")
print("-" * 72)
print(f"Caracteres que el tokenizador no puede representar: {perdidos}")

In [ ]:
def tasa_unk(tok, textos):
    total = unk = 0
    for t in textos:
        ids = tok(t, add_special_tokens=False)["input_ids"]
        total += len(ids)
        unk += sum(1 for i in ids if i == tok.unk_token_id)
    return unk / max(total, 1), total


textos_crudos = [r["entrada"] + " " + r["salida"] for r in registros]
tasa_antes, n_tokens_antes = tasa_unk(tokenizer, textos_crudos)
print(f"Corpus SIN normalizar: {tasa_antes:.4%} de tokens <unk> "
      f"sobre {n_tokens_antes} tokens")

afectados = sum(1 for t in textos_crudos
                if tokenizer.unk_token_id in tokenizer(t, add_special_tokens=False)["input_ids"])
print(f"Ejemplos con al menos un <unk>: {afectados} de {len(textos_crudos)} "
      f"({afectados/len(textos_crudos):.1%})")

### La solución: normalización dirigida

Dos caminos posibles:

| Opción | Cómo | Por qué no / por qué sí |
|---|---|---|
| **Ampliar el vocabulario** | `tokenizer.add_tokens([...])` + `model.resize_token_embeddings()` | Los embeddings de los tokens nuevos nacen aleatorios y solo se entrenan con nuestros 132 ejemplos. Un símbolo que aparece 20 veces no alcanza a aprender un embedding útil. Con un corpus grande sería la opción correcta. |
| **Normalizar el texto** ✔ | `÷` → `/`, `×` → `x`, `√` → `raiz`, `²` → `^2` | Reescribe la notación usando caracteres que el modelo **ya entiende bien** por su preentrenamiento en inglés: `/`, `x`, `^` aparecen en matemáticas en cualquier idioma. Sin parámetros nuevos que entrenar. |

Elegimos normalizar. La condición para que sea válido es que la
transformación **preserve el significado**: `864 ÷ 12` y `864 / 12` son la
misma operación, y `√144` y `raiz(144)` también.

Coste que hay que declarar: la normalización se aplica también a la salida, así
que este modelo genera "105 estudiantes" sin tildes y con `x` en lugar de `×`.
Es una diferencia estética frente a Qwen. Para las métricas no importa —ROUGE-L
normaliza tildes y la exactitud compara números—, pero para un producto real
haría falta un paso de post-proceso que devuelva la notación.

In [ ]:
import re
import unicodedata

REEMPLAZOS = {
    "×": " x ", "·": " x ", "÷": " / ", "−": "-", "–": "-", "—": "-",
    "²": "^2", "³": "^3", "⁴": "^4", "√": " raiz ",
    "°": " grados ", "π": " pi ", "≈": " ~ ", "≤": " <= ", "≥": " >= ",
    "¿": "", "¡": "", "“": '"', "”": '"', "’": "'",
}


def normalizar_para_t5(texto, eliminar_tildes=None):
    """Reescribe el texto dentro del vocabulario que FLAN-T5 sí representa."""
    if eliminar_tildes is None:
        eliminar_tildes = ELIMINAR_TILDES
    for viejo, nuevo in REEMPLAZOS.items():
        texto = texto.replace(viejo, nuevo)
    if eliminar_tildes:
        # NFKD separa la letra de su tilde; descartamos las marcas combinantes.
        # La 'ñ' se trataría igual, así que se protege antes.
        texto = texto.replace("ñ", "@N@").replace("Ñ", "@NN@")
        texto = unicodedata.normalize("NFKD", texto)
        texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
        texto = texto.replace("@N@", "ni").replace("@NN@", "NI")
    return re.sub(r"\s+", " ", texto).strip()


for ejemplo in ["¿Cuál es el área de un círculo? √144 ÷ 2 = 6",
                "Paso 1: Calculamos 5² × 3 y la división 864 ÷ 12."]:
    print("antes :", ejemplo)
    print("después:", normalizar_para_t5(ejemplo))
    print()

In [ ]:
textos_norm = [normalizar_para_t5(t) for t in textos_crudos]
tasa_despues, n_tokens_despues = tasa_unk(tokenizer, textos_norm)

print(f"Tasa de <unk> ANTES  : {tasa_antes:.4%}")
print(f"Tasa de <unk> DESPUÉS: {tasa_despues:.4%}")
print(f"Tokens totales: {n_tokens_antes} -> {n_tokens_despues} "
      f"({100*(n_tokens_despues-n_tokens_antes)/n_tokens_antes:+.1f}%)")

DIAGNOSTICO_TOKENIZADOR = {
    "unk_antes": tasa_antes,
    "unk_despues": tasa_despues,
    "tokens_antes": n_tokens_antes,
    "tokens_despues": n_tokens_despues,
    "caracteres_perdidos": perdidos,
    "eliminar_tildes": ELIMINAR_TILDES,
}

> **Cómo leer el cambio en el número de tokens.** Si tras normalizar hay *más*
> tokens, no es un fallo: `√144` era un `<unk>` y un número (2 tokens, uno de
> ellos inútil); `raiz(144)` son varios tokens **todos informativos**. Se paga
> longitud a cambio de información. Ese intercambio es exactamente el que un
> tokenizador mal ajustado al idioma impone, y es el argumento cuantitativo de
> la comparación de tokenizadores del notebook 5.

In [ ]:
import numpy as np

entradas_tok = [tokenizer(normalizar_para_t5(construir_entrada(r)))["input_ids"] for r in registros]
salidas_tok  = [tokenizer(normalizar_para_t5(construir_objetivo(r)))["input_ids"] for r in registros]
le, ls = np.array([len(x) for x in entradas_tok]), np.array([len(x) for x in salidas_tok])

print(f"Entrada : media={le.mean():.1f}  p95={np.percentile(le,95):.0f}  max={le.max()}"
      f"   -> truncados con LONGITUD_ENTRADA={LONGITUD_ENTRADA}: {(le>LONGITUD_ENTRADA).sum()}")
print(f"Salida  : media={ls.mean():.1f}  p95={np.percentile(ls,95):.0f}  max={ls.max()}"
      f"   -> truncados con LONGITUD_SALIDA={LONGITUD_SALIDA}: {(ls>LONGITUD_SALIDA).sum()}")

fertilidades = [len(tokenizer(normalizar_para_t5(r["entrada"] + " " + r["salida"]),
                              add_special_tokens=False)["input_ids"])
                / len((r["entrada"] + " " + r["salida"]).split()) for r in registros]

STATS_TOKENIZADOR = {
    "modelo": MODELO_ID,
    "tokenizador": tokenizer.__class__.__name__,
    "vocabulario": int(tokenizer.vocab_size),
    "fertilidad_media": float(np.mean(fertilidades)),
    "tokens_media": float(le.mean() + ls.mean()),
    "tokens_max": int(le.max() + ls.max()),
    **DIAGNOSTICO_TOKENIZADOR,
}
print(f"\nFertilidad (tokens/palabra, texto normalizado): {np.mean(fertilidades):.3f}")

### Tokenización del dataset

En seq2seq no hace falta enmascarar el prompt: entrada y salida van a
subredes distintas, y la pérdida se calcula solo sobre el decoder. Lo que sí
hay que hacer es sustituir el padding de las etiquetas por `-100` para que no
contribuya a la pérdida.

In [ ]:
from datasets import Dataset


def tokenizar(reg):
    entrada = normalizar_para_t5(construir_entrada(reg))
    objetivo = normalizar_para_t5(construir_objetivo(reg))

    tok = tokenizer(entrada, truncation=True, max_length=LONGITUD_ENTRADA)
    etiquetas = tokenizer(text_target=objetivo, truncation=True,
                          max_length=LONGITUD_SALIDA)["input_ids"]
    # El padding de las etiquetas debe ignorarse en la pérdida.
    tok["labels"] = [t if t != tokenizer.pad_token_id else -100 for t in etiquetas]
    return tok


ds_train = Dataset.from_list([tokenizar(r) for r in train])
ds_val   = Dataset.from_list([tokenizar(r) for r in val])
print(ds_train)

ej = ds_train[0]
print("\nEntrada :", tokenizer.decode(ej["input_ids"], skip_special_tokens=True))
print("Objetivo:", tokenizer.decode([t for t in ej["labels"] if t != -100],
                                    skip_special_tokens=True))

## 7 · Baseline

FLAN-T5 fue afinado con instrucciones, así que responde a "resuelve el problema
paso a paso" sin haber visto nuestro corpus. Esperen dos comportamientos
característicos del baseline:

- **Respuestas muy cortas.** FLAN fue entrenado mayoritariamente con tareas de
  respuesta breve (clasificación, preguntas de una palabra). Su sesgo natural
  es contestar con dos palabras, no con un procedimiento.
- **Ninguna línea `Categoria:`.** Nadie le ha enseñado ese formato todavía. La
  accuracy de clasificación del baseline será prácticamente 0, y eso no
  significa que no entienda el problema: significa que no conoce el formato.

Ambas cosas quedan registradas por separado gracias a `formato_valido`.

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

modelo = AutoModelForSeq2SeqLM.from_pretrained(MODELO_ID).to(DEVICE)

n_par = sum(p.numel() for p in modelo.parameters())
print(f"Modelo: {MODELO_ID}")
print(f"Parámetros: {n_par/1e6:.1f} M")
print(f"Bloques encoder: {modelo.config.num_layers} | "
      f"bloques decoder: {modelo.config.num_decoder_layers} | "
      f"d_model: {modelo.config.d_model}")

In [ ]:
@torch.no_grad()
def generar(modelo, entradas, max_new_tokens=MAX_TOKENS_GEN, batch=8):
    """Genera en lotes: T5 admite padding a la izquierda o derecha sin problema
    porque el encoder usa máscara de atención explícita."""
    modelo.eval()
    salidas = []
    for i in tqdm(range(0, len(entradas), batch), desc="generando"):
        textos = [normalizar_para_t5(PREFIJO + e) for e in entradas[i:i + batch]]
        lote = tokenizer(textos, return_tensors="pt", padding=True,
                         truncation=True, max_length=LONGITUD_ENTRADA).to(modelo.device)
        out = modelo.generate(**lote, max_new_tokens=max_new_tokens,
                              do_sample=False, num_beams=1)
        salidas.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return [s.strip() for s in salidas]


gen_baseline = generar(modelo, [r["entrada"] for r in val])
metricas_baseline = evaluar_generacion(gen_baseline, val)

print()
for k, v in metricas_baseline.items():
    if not k.startswith("_"):
        print(f"  {k:16s}: {v}")

In [ ]:
def accuracy_categoria(generados, registros):
    """Accuracy de la clasificación implícita, extraída de la línea 'Categoria:'."""
    aciertos = [categoria_predicha(g) == r["categoria"] for g, r in zip(generados, registros)]
    detectadas = sum(1 for g in generados if categoria_predicha(g) is not None)
    return {
        "accuracy": sum(aciertos) / len(aciertos),
        "categoria_presente": detectadas / len(generados),
        "_aciertos": aciertos,
    }


clf_baseline = accuracy_categoria(gen_baseline, val)
print(f"Clasificación implícita (baseline): accuracy={clf_baseline['accuracy']:.1%}  "
      f"línea 'Categoria:' presente en {clf_baseline['categoria_presente']:.1%} de las salidas")

In [ ]:
gen_demo_baseline = generar(modelo, [d["entrada"] for d in demo])
for d, g in zip(demo, gen_demo_baseline):
    print("=" * 78)
    print(f"[{d['id']} · {d['categoria']}] {d['entrada']}")
    print("-" * 78)
    print("BASELINE:", g[:500] if g else "(salida vacía)")
    print(f"¿Correcta? {respuesta_correcta(g, d['valor'])}  "
          f"¿Formato? {formato_valido(g)}")
print("=" * 78)

## 8 · Configuración del fine-tuning

### Una advertencia específica de T5: nada de fp16

Los notebooks 1 y 2 entrenan en precisión mixta fp16. **Aquí no.** T5 fue
preentrenado en bfloat16, un formato con el mismo rango exponencial que fp32
pero menos precisión. Sus activaciones internas alcanzan magnitudes que en
fp16 **desbordan a infinito**, y la pérdida se convierte en `NaN` a los pocos
pasos. Es un problema conocido y documentado de la familia T5.

La T4 de Colab no soporta bfloat16 por hardware, así que la opción correcta es
entrenar en fp32. Con 250M de parámetros y LoRA cabe sin dificultad, y de
hecho es más rápido que depurar `NaN`s.

Esta diferencia obligada entre arquitecturas es en sí misma un resultado del
laboratorio: reportarla es parte de la comparación de "facilidad de
fine-tuning".

### LoRA en T5

`target_modules=["q", "v"]` alcanza los tres tipos de atención del modelo
(encoder, decoder y cross-attention), porque las tres usan proyecciones con
esos nombres. Rango 16 y alpha 32, igual que en Qwen, para que la comparación
entre arquitecturas no esté contaminada por hiperparámetros distintos.

In [ ]:
import inspect
from transformers import TrainingArguments


def construir_args(clase=TrainingArguments, **kwargs):
    """Crea TrainingArguments filtrando los parámetros que la versión instalada
    de `transformers` no reconoce.

    Motivo: entre versiones recientes `evaluation_strategy` pasó a llamarse
    `eval_strategy`. Sin esta capa, el notebook funciona hoy y falla el
    semestre que viene. Se prefiere `eval_strategy` y se traduce si hace falta.
    """
    admitidos = set(inspect.signature(clase.__init__).parameters)
    if "eval_strategy" in kwargs and "eval_strategy" not in admitidos:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    descartados = [k for k in kwargs if k not in admitidos]
    for k in descartados:
        kwargs.pop(k)
    if descartados:
        print(f"Parámetros no soportados por esta versión, se omiten: {descartados}")
    return clase(**kwargs)

In [ ]:
from peft import LoraConfig, get_peft_model

config_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],    # cubre encoder, decoder y cross-attention
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

modelo = get_peft_model(modelo, config_lora)
modelo.print_trainable_parameters()

### Weights & Biases

W&B registra automáticamente la curva de pérdida, los hiperparámetros y el
consumo de GPU. Es lo que después nos permitirá comparar las cuatro
arquitecturas sobre los mismos ejes en lugar de sobre capturas de pantalla.

Convención de nombres del proyecto (idéntica en los cinco notebooks):

- **Proyecto:** `tutor-matematicas-arquitecturas`
- **Run:** `flan-t5-lora-finetune`
- **Tags:** identifican arquitectura y fase, para poder filtrar en el panel

Si no quieren usar W&B, pongan `USAR_WANDB = False`: el notebook seguirá
funcionando y las métricas se guardarán igual en `resultados/`.

In [ ]:
import os

USAR_WANDB   = True          # ponlo en False para trabajar sin conexión a W&B
PROYECTO     = "tutor-matematicas-arquitecturas"
NOMBRE_RUN   = "flan-t5-lora-finetune"
TAGS         = ["flan-t5", "encoder-decoder", "lora", "generacion", "clasificacion"]

if USAR_WANDB:
    import wandb
    wandb.login()            # pedirá la API key la primera vez
    os.environ["WANDB_PROJECT"] = PROYECTO
    os.environ["WANDB_LOG_MODEL"] = "false"
    REPORTAR_A = "wandb"
else:
    os.environ["WANDB_MODE"] = "disabled"
    REPORTAR_A = "none"

print(f"Registro de experimentos: {REPORTAR_A}  |  run: {NOMBRE_RUN}")

In [ ]:
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=modelo,               # construye decoder_input_ids desplazando labels
    label_pad_token_id=-100,
    padding=True,
    return_tensors="pt",
)

args = construir_args(
    clase=Seq2SeqTrainingArguments,
    output_dir=f"{DIR_CHECKPOINTS}/flan-t5-lora",
    run_name=NOMBRE_RUN,

    num_train_epochs=15,        # más que Qwen: el modelo es menor y parte de más lejos
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,

    learning_rate=3e-4,         # T5 con LoRA tolera algo más que Qwen
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,

    fp16=False,                 # IMPRESCINDIBLE: fp16 rompe T5 (ver arriba)
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to=REPORTAR_A,
    seed=SEMILLA,
    predict_with_generate=False,   # la generación se evalúa aparte, con control total
)

trainer = Seq2SeqTrainer(
    model=modelo,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
)

pasos_epoca = len(ds_train) // (args.per_device_train_batch_size * args.gradient_accumulation_steps)
print(f"Pasos de optimización por época: {pasos_epoca}")
print(f"Pasos totales: {int(args.num_train_epochs) * pasos_epoca}")

## 9 · Entrenamiento

Qué esperar, y en qué se diferencia de las curvas de Qwen:

- La pérdida inicial será **más alta** que la de Qwen. Es normal: T5 no solo
  aprende el formato de la respuesta, también aprende a emitir la línea
  `Categoria:`, que es una tarea adicional.
- El descenso suele ser **más suave y sostenido**. Un modelo de 250M con LoRA
  tiene menos capacidad para memorizar 132 ejemplos de golpe, así que el
  sobreajuste aparece más tarde que en Qwen.
- Si ven `nan` en la pérdida, revisen que `fp16=False`. Es la causa en la
  práctica totalidad de los casos.

In [ ]:
resultado_entrenamiento = trainer.train()
print()
print(f"Tiempo de entrenamiento: {resultado_entrenamiento.metrics['train_runtime']:.1f} s")
print(f"Pérdida final: {resultado_entrenamiento.metrics['train_loss']:.4f}")

In [ ]:
import pandas as pd

historial = pd.DataFrame(trainer.state.log_history)
ev = historial.dropna(subset=["eval_loss"])
print(ev[["epoch", "eval_loss"]].to_string(index=False))

mejor = ev.sort_values("eval_loss").iloc[0]
print(f"\nMejor época: {mejor['epoch']:.0f}  (eval_loss = {mejor['eval_loss']:.4f})")

In [ ]:
import matplotlib.pyplot as plt

tr = historial.dropna(subset=["loss"])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(tr["epoch"], tr["loss"], label="Entrenamiento", alpha=0.8)
ax.plot(ev["epoch"], ev["eval_loss"], label="Validación", marker="o")
ax.axvline(mejor["epoch"], ls="--", c="gray", lw=1, label=f"Mejor época ({mejor['epoch']:.0f})")
ax.set_xlabel("Época"); ax.set_ylabel("Pérdida")
ax.set_title("FLAN-T5-base + LoRA · curvas de pérdida")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Tras `load_best_model_at_end` el modelo bueno es `trainer.model`: el Trainer
# recarga el mejor checkpoint, y evaluar sobre `modelo` usaría la última época.
modelo = trainer.model

modelo.save_pretrained(DIR_ADAPTADOR)
tokenizer.save_pretrained(DIR_ADAPTADOR)
print(f"Adaptador guardado en {DIR_ADAPTADOR}/")

## 10 · Integración con Weights & Biases

Mismo proyecto y misma convención que los otros tres notebooks, para que las
curvas sean superponibles en el panel:

- **Proyecto:** `tutor-matematicas-arquitecturas`
- **Run:** `flan-t5-lora-finetune`
- **Tags:** `flan-t5`, `encoder-decoder`, `lora`, `generacion`, `clasificacion`

Lo específico de este notebook que se registra a mano: el diagnóstico del
tokenizador (tasa de `<unk>` antes y después de normalizar) y la accuracy de
clasificación implícita, que es lo que permite comparar este modelo contra
BETO y contra el pipeline.

## 11 · Evaluación

Evaluamos **las dos capacidades por separado**, porque el modelo hace dos
trabajos y un promedio conjunto no diría nada útil:

1. **Generación** — mismas métricas que los notebooks 1 y 3.
2. **Clasificación implícita** — accuracy sobre la línea `Categoria:`,
   directamente comparable con el notebook 2.

In [ ]:
gen_finetuned = generar(modelo, [r["entrada"] for r in val])
metricas_finetuned = evaluar_generacion(gen_finetuned, val)
clf_finetuned = accuracy_categoria(gen_finetuned, val)

print("Generación")
for k, v in metricas_finetuned.items():
    if not k.startswith("_"):
        print(f"  {k:16s}: {v}")
print()
print("Clasificación implícita")
print(f"  accuracy            : {clf_finetuned['accuracy']:.1%}")
print(f"  categoría presente  : {clf_finetuned['categoria_presente']:.1%}")

### ¿Se ayudan las dos tareas entre sí?

La hipótesis detrás de escribir la categoría primero es que condiciona
favorablemente la generación posterior. Se puede comprobar: comparen la
exactitud de la respuesta en los ejemplos donde el modelo clasificó bien contra
aquellos donde clasificó mal.

Es el mismo análisis de propagación de error del notebook 3, pero **dentro de
un solo modelo**. Y por eso mismo hay que interpretarlo con cuidado: aquí no
hay causalidad garantizada. Un ejemplo difícil puede provocar a la vez una
clasificación errónea y una respuesta errónea, sin que una cause la otra.

In [ ]:
ok_clf = clf_finetuned["_aciertos"]
ok_gen = metricas_finetuned["_correctas"]

g_ok  = [g for g, c in zip(ok_gen, ok_clf) if c]
g_mal = [g for g, c in zip(ok_gen, ok_clf) if not c]

print("Coherencia entre las dos tareas (modelo integrado)")
print("=" * 66)
print(f"Clasificó BIEN  n={len(g_ok):2d}  exactitud de la respuesta = "
      f"{sum(g_ok)/max(len(g_ok),1):.1%}")
print(f"Clasificó MAL   n={len(g_mal):2d}  exactitud de la respuesta = "
      f"{sum(g_mal)/max(len(g_mal),1):.1%}")
print("=" * 66)
if len(g_mal) < 5:
    print(f"AVISO: solo {len(g_mal)} ejemplos mal clasificados; la cifra es anecdótica.")

In [ ]:
from collections import defaultdict

por_cat = defaultdict(lambda: {"n": 0, "base": 0, "ft": 0})
for r, b, f in zip(val, metricas_baseline["_correctas"], metricas_finetuned["_correctas"]):
    d = por_cat[r["categoria"]]
    d["n"] += 1; d["base"] += int(b); d["ft"] += int(f)

print(f"{'categoría':26s} {'n':>3s} {'baseline':>10s} {'fine-tuned':>12s}")
print("-" * 56)
for cat in CATEGORIAS:
    d = por_cat.get(cat)
    if d:
        print(f"{cat:26s} {d['n']:3d} {d['base']/d['n']:>9.0%} {d['ft']/d['n']:>11.0%}")

EXACTITUD_POR_CATEGORIA = {
    cat: {"n": d["n"], "baseline": d["base"]/d["n"], "finetuned": d["ft"]/d["n"]}
    for cat, d in por_cat.items()
}

## 12 · Comparación Baseline vs Fine-tuned

In [ ]:
tabla_metricas(metricas_baseline, metricas_finetuned,
               titulo=f"FLAN-T5-base · {len(val)} ejemplos de validación")

print()
print(f"Clasificación implícita: {clf_baseline['accuracy']:.1%} -> "
      f"{clf_finetuned['accuracy']:.1%}  "
      f"({clf_finetuned['accuracy'] - clf_baseline['accuracy']:+.1%})")

In [ ]:
gen_demo_ft = generar(modelo, [d["entrada"] for d in demo])

for d, antes, despues in zip(demo, gen_demo_baseline, gen_demo_ft):
    print("=" * 78)
    print(f"[{d['id']} · {d['categoria']}] {d['entrada']}")
    print("-" * 78)
    print("ANTES:", antes[:400] if antes else "(vacío)")
    print("-" * 78)
    print("DESPUÉS:")
    print(despues[:500])
    print("-" * 78)
    print(f"Categoría predicha: {categoria_predicha(despues)} (real: {d['categoria']})")
    print(f"Correcta -> antes: {respuesta_correcta(antes, d['valor'])} | "
          f"después: {respuesta_correcta(despues, d['valor'])}")
print("=" * 78)

In [ ]:
RESULTADOS_T5 = {
    "notebook": "S04_Lab_Fine_tuning_FLAN_T5",
    "arquitectura": "encoder-decoder",
    "modelo": MODELO_ID,
    "metodo": "LoRA",
    "n_train": len(train),
    "n_val": len(val),
    "epocas": float(args.num_train_epochs),
    "learning_rate": args.learning_rate,
    "precision": "fp32 (fp16 desborda en T5)",
    "tiempo_entrenamiento_s": resultado_entrenamiento.metrics["train_runtime"],
    "train_loss_final": resultado_entrenamiento.metrics["train_loss"],
    "eval_loss_mejor": float(mejor["eval_loss"]),
    "mejor_epoca": float(mejor["epoch"]),
    "parametros_entrenables": sum(p.numel() for p in modelo.parameters() if p.requires_grad),
    "parametros_totales": n_par,
    "baseline": metricas_baseline,
    "finetuned": metricas_finetuned,
    "clasificacion_baseline": {k: v for k, v in clf_baseline.items() if not k.startswith("_")},
    "clasificacion_finetuned": {k: v for k, v in clf_finetuned.items() if not k.startswith("_")},
    "coherencia_tareas": {
        "n_clf_ok": len(g_ok),
        "exactitud_clf_ok": sum(g_ok) / max(len(g_ok), 1),
        "n_clf_mal": len(g_mal),
        "exactitud_clf_mal": sum(g_mal) / max(len(g_mal), 1),
    },
    "por_categoria": EXACTITUD_POR_CATEGORIA,
    "tokenizador": STATS_TOKENIZADOR,
}

guardar_resultados("flan_t5", RESULTADOS_T5)

In [ ]:
if USAR_WANDB:
    import wandb

    if wandb.run is None:
        wandb.init(project=PROYECTO, name=NOMBRE_RUN, tags=TAGS, reinit=True)

    tabla = wandb.Table(columns=["id", "entrada", "referencia", "baseline", "finetuned",
                                 "cat_real", "cat_predicha", "ok_baseline", "ok_finetuned"])
    for r, b, f, okb, okf in zip(val, gen_baseline, gen_finetuned,
                                 metricas_baseline["_correctas"],
                                 metricas_finetuned["_correctas"]):
        tabla.add_data(r["id"], r["entrada"], r["salida"], b, f,
                       r["categoria"], categoria_predicha(f), okb, okf)
    wandb.log({"evaluacion/generaciones": tabla})

    wandb.summary.update({
        "tokenizador/unk_antes": tasa_antes,
        "tokenizador/unk_despues": tasa_despues,
        "baseline/exactitud": metricas_baseline["exactitud"],
        "baseline/formato_valido": metricas_baseline["formato_valido"],
        "baseline/clf_accuracy": clf_baseline["accuracy"],
        "finetuned/exactitud": metricas_finetuned["exactitud"],
        "finetuned/formato_valido": metricas_finetuned["formato_valido"],
        "finetuned/clf_accuracy": clf_finetuned["accuracy"],
        "delta/exactitud": metricas_finetuned["exactitud"] - metricas_baseline["exactitud"],
    })
    wandb.finish()
    print("Registro en W&B completado.")
else:
    print("W&B desactivado; las métricas quedaron en resultados/flan_t5.json")

## 13 · Discusión

**1. ¿Cuánto costó el tokenizador?**
Es la pregunta propia de este notebook. Con la tasa de `<unk>` antes y después
de normalizar pueden estimar cuánta información se habría perdido sin
tratamiento. Y aun normalizado, comparen la fertilidad (tokens por palabra) con
la de Qwen: si FLAN-T5 necesita bastantes más tokens para el mismo texto en
español, cada ejemplo consume más contexto y más cómputo por la sola razón de
que el vocabulario no fue diseñado para este idioma.

**2. ¿Un modelo integrado o dos especializados?**
Contrasten la exactitud de este notebook contra la configuración B del notebook
3, y la accuracy de clasificación contra BETO. Los escenarios posibles:

| Resultado | Lectura |
|---|---|
| T5 gana en ambas | El modelo integrado domina. La simplicidad operativa se suma a la calidad: decisión fácil. |
| T5 clasifica peor pero genera igual | Esperable: BETO es un especialista con atención bidireccional pura y una cabeza dedicada. La pregunta pasa a ser si la clasificación importa para el resultado final. |
| T5 pierde en ambas | Probablemente sea cuestión de tamaño (250M vs 1.5B) y de tokenizador, no de arquitectura. Habría que repetirlo con `flan-t5-large` o con mT5 para separar los factores. |

**3. ¿La clasificación implícita ayuda a generar?**
Miren el análisis de coherencia. Y recuerden la advertencia: correlación no es
causalidad, ambos errores pueden compartir causa (un problema difícil).

**4. ¿Qué limita a este modelo?**
Tres factores confundidos que conviene nombrar por separado: tamaño (250M),
tokenizador (inglés), y preentrenamiento matemático (más débil que Qwen). Con
un solo experimento no se pueden separar, y decirlo es más honesto que atribuir
el resultado a "la arquitectura encoder-decoder".

## 14 · Conclusiones

1. **El tokenizador es un criterio de selección de modelo, no un detalle de
   implementación.** Es la lección más transferible del laboratorio: antes de
   elegir un modelo para un idioma o dominio concreto, midan cómo tokeniza sus
   datos. Cuesta cinco minutos y puede ahorrar un experimento entero.
2. **Un modelo integrado elimina toda una clase de fallos.** Sin desalineación
   de etiquetas, sin propagación en cascada, sin dos artefactos que versionar.
3. **La arquitectura encoder-decoder es conceptualmente la adecuada** para
   entrada estructurada → salida estructurada. Si pierde en las métricas, hay
   que verificar si es por la arquitectura o por el modelo concreto elegido.
4. **Todo lo anterior se cuantifica en el notebook 5**, que agrega los cuatro
   `resultados/*.json` y produce la comparación final.